# Design Case: General Examples

Paul T. Grogan <paul.grogan@asu.edu>

What is the mean revisit period over Tempe Arizona for the VIIRS instrument onboard NOAA 20?

In [ ]:
from tatc.schemas import Satellite, TwoLineElements, Instrument, Point
from tatc.utils import swath_width_to_field_of_regard

# from https://celestrak.org/NORAD/elements/gp.php?FORMAT=tle&NAME=NOAA%2020
orbit = TwoLineElements(
    tle=[
        "1 43013U 17073A   26022.94240312  .00000200  00000+0  11574-3 0  9998",
        "2 43013  98.7645 323.7982 0001179  28.0279 332.0961 14.19539020423863"
    ]
)
# from https://space.oscar.wmo.int/instruments/view/viirs
altitude_m = orbit.get_altitude()
swath_width_m = 3000e3
instrument = Instrument(
    name="VIIRS",
    field_of_regard=swath_width_to_field_of_regard(altitude_m, swath_width_m)
)
satellite = Satellite(
    name="NOAA 20",
    orbit=orbit,
    instruments=[instrument]
)

# from google
point = Point(id=0, latitude=33.4255, longitude=-111.9400)

from tatc.analysis import collect_observations, aggregate_observations, reduce_observations
from datetime import timedelta
start = orbit.get_epoch()
end = start + timedelta(days=30)
raw_observations = collect_observations(point, satellite, start, end)
aggregated_observations = aggregate_observations(raw_observations)
reduced_observations = reduce_observations(aggregated_observations)

mean_revisit_hr = reduced_observations.iloc[0].revisit/timedelta(hours=1)
display(f"mean revisit period: {mean_revisit_hr:0.1f} hours")

What is the mean revisit period over Tempe Arizona for a VIIRS instrument with a Walker Delta constellation with 3 satellites in 3 planes following the orbit of NOAA 20?

In [ ]:
from tatc.schemas import WalkerConstellation, TwoLineElements, Instrument, Point
from tatc.utils import swath_width_to_field_of_regard

# from https://celestrak.org/NORAD/elements/gp.php?FORMAT=tle&NAME=NOAA%2020
orbit = TwoLineElements(
    tle=[
        "1 43013U 17073A   26022.94240312  .00000200  00000+0  11574-3 0  9998",
        "2 43013  98.7645 323.7982 0001179  28.0279 332.0961 14.19539020423863"
    ]
)
# from https://space.oscar.wmo.int/instruments/view/viirs
altitude_m = orbit.get_altitude()
swath_width_m = 3000e3
instrument = Instrument(
    name="VIIRS",
    field_of_regard=swath_width_to_field_of_regard(altitude_m, swath_width_m)
)
constellation = WalkerConstellation(
    name="NOAA 20",
    orbit=orbit,
    instruments=[instrument],
    configuration="delta",
    number_satellites=3,
    number_planes=3
)

# from google
point = Point(id=0, latitude=33.4255, longitude=-111.9400)

from tatc.analysis import collect_multi_observations, aggregate_observations, reduce_observations
from datetime import timedelta
start = orbit.get_epoch()
end = start + timedelta(days=30)
raw_observations = collect_multi_observations(point, constellation.generate_members(), start, end)
aggregated_observations = aggregate_observations(raw_observations)
reduced_observations = reduce_observations(aggregated_observations)

mean_revisit_hr = reduced_observations.iloc[0].revisit/timedelta(hours=1)
display(f"mean revisit period: {mean_revisit_hr:0.1f} hours")

How does mean revisit period over Tempe Arizona change with 1-6 satellites per plane for a VIIRS instrument onboard a Walker Delta constellation with 3 planes following the orbit of NOAA 20?

In [ ]:
from tatc.schemas import WalkerConstellation, TwoLineElements, Instrument, Point
from tatc.utils import swath_width_to_field_of_regard

# from https://celestrak.org/NORAD/elements/gp.php?FORMAT=tle&NAME=NOAA%2020
orbit = TwoLineElements(
    tle=[
        "1 43013U 17073A   26022.94240312  .00000200  00000+0  11574-3 0  9998",
        "2 43013  98.7645 323.7982 0001179  28.0279 332.0961 14.19539020423863"
    ]
)
# from https://space.oscar.wmo.int/instruments/view/viirs
altitude_m = orbit.get_altitude()
swath_width_m = 3000e3
instrument = Instrument(
    name="VIIRS",
    field_of_regard=swath_width_to_field_of_regard(altitude_m, swath_width_m)
)

# from google
point = Point(id=0, latitude=33.4255, longitude=-111.9400)

number_satellites_per_plane = range(1,7)
mean_revisit_hr = [None]*len(number_satellites_per_plane)
for i, x in enumerate(number_satellites_per_plane):
    constellation = WalkerConstellation(
        name="NOAA 20",
        orbit=orbit,
        instruments=[instrument],
        configuration="delta",
        number_satellites=3*x,
        number_planes=3
    )

    from tatc.analysis import collect_multi_observations, aggregate_observations, reduce_observations
    from datetime import timedelta
    start = orbit.get_epoch()
    end = start + timedelta(days=30)
    raw_observations = collect_multi_observations(point, constellation.generate_members(), start, end)
    aggregated_observations = aggregate_observations(raw_observations)
    reduced_observations = reduce_observations(aggregated_observations)
    mean_revisit_hr[i] = reduced_observations.iloc[0].revisit/timedelta(hours=1)

for (x, t) in zip(number_satellites_per_plane, mean_revisit_hr):
    display(f"mean revisit period for {x} satellites per plane: {t:0.2f} hours")

What is the ground track for the VIIRS instrument onboard NOAA 20 over a 30-minute period?

In [ ]:
from tatc.schemas import Satellite, TwoLineElements, Instrument, Point
from tatc.utils import swath_width_to_field_of_regard

# from https://celestrak.org/NORAD/elements/gp.php?FORMAT=tle&NAME=NOAA%2020
orbit = TwoLineElements(
    tle=[
        "1 43013U 17073A   26022.94240312  .00000200  00000+0  11574-3 0  9998",
        "2 43013  98.7645 323.7982 0001179  28.0279 332.0961 14.19539020423863"
    ]
)
# from https://space.oscar.wmo.int/instruments/view/viirs
altitude_m = orbit.get_altitude()
swath_width_m = 3000e3
instrument = Instrument(
    name="VIIRS",
    field_of_regard=swath_width_to_field_of_regard(altitude_m, swath_width_m)
)
satellite = Satellite(
    name="NOAA 20",
    orbit=orbit,
    instruments=[instrument]
)

from tatc.analysis import compute_ground_track
from tatc.utils.orbital import compute_ground_surface_velocity
from datetime import timedelta
import pandas as pd
start = orbit.get_epoch()
end = start + timedelta(minutes=30)
inclination_deg = orbit.get_inclination()
max_delta_t_s = swath_width_m / compute_ground_surface_velocity(altitude_m, inclination_deg)
delta_t = timedelta(seconds=max_delta_t_s / 10)
times = pd.date_range(start, end, freq=delta_t)
ground_track = compute_ground_track(satellite, times)

display(f"ground track polygon: {ground_track.iloc[0].geometry}")

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
fig, ax = plt.subplots(figsize=(8, 5), subplot_kw={"projection": ccrs.PlateCarree()})

ground_track.plot(
    ax=ax,
    transform=ccrs.PlateCarree()
)
ax.coastlines()
ax.set_global()
plt.show()

Perform a global coverage analysis with mean revisit period for the VIIRS instrument onboard NOAA 20 for sample points equally spaced at 5000 km. 

In [ ]:
from tatc.schemas import WalkerConstellation, TwoLineElements, Instrument, Point
from tatc.utils import swath_width_to_field_of_regard

# from https://celestrak.org/NORAD/elements/gp.php?FORMAT=tle&NAME=NOAA%2020
orbit = TwoLineElements(
    tle=[
        "1 43013U 17073A   26022.94240312  .00000200  00000+0  11574-3 0  9998",
        "2 43013  98.7645 323.7982 0001179  28.0279 332.0961 14.19539020423863"
    ]
)
# from https://space.oscar.wmo.int/instruments/view/viirs
altitude_m = orbit.get_altitude()
swath_width_m = 3000e3
instrument = Instrument(
    name="VIIRS",
    field_of_regard=swath_width_to_field_of_regard(altitude_m, swath_width_m)
)
constellation = WalkerConstellation(
    name="NOAA 20",
    orbit=orbit,
    instruments=[instrument],
    configuration="delta",
    number_satellites=3,
    number_planes=3
)

from tatc.generation import generate_points_uniform_spacing
mean_distance_m = 5000e3
points_df = generate_points_uniform_spacing(mean_distance_m)
points = points_df.apply(lambda r: Point(id=r.point_id, latitude=r.geometry.y, longitude=r.geometry.x), axis=1)

from tatc.analysis import collect_multi_observations, aggregate_observations, reduce_observations
from datetime import timedelta
start = orbit.get_epoch()
end = start + timedelta(days=30)
raw_observations = pd.concat([
    collect_multi_observations(point, constellation.generate_members(), start, end)
    for point in points
])
aggregated_observations = aggregate_observations(raw_observations)
reduced_observations = reduce_observations(aggregated_observations)

reduced_observations["mean_revisit_hr"] = reduced_observations.apply(
    lambda r: r["revisit"] / timedelta(hours=1), axis=1
)
display(reduced_observations[["point_id", "geometry", "mean_revisit_hr"]])

import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={"projection": ccrs.PlateCarree()})
reduced_observations.plot(
    column="mean_revisit_hr",
    cmap="viridis",
    legend=True,
    legend_kwds={"label": "Revisit (hr)", "orientation": "horizontal"},
    ax=ax,
    transform=ccrs.PlateCarree()
)
ax.coastlines()
plt.show()